In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
chunksize = 50000
sample_fraction = 0.1
samples = []

for chunk in pd.read_csv('../data/bosch_subset/bosch-production-line-performance/train_numeric.csv/train_numeric.csv', chunksize=chunksize):
    sample = chunk.sample(frac=sample_fraction, random_state=42)
    samples.append(sample)

df_bosch = pd.concat(samples, ignore_index=True)
print(df_bosch.shape)

(118375, 970)


In [5]:
df_bosch.columns[:20].tolist()

['Id',
 'L0_S0_F0',
 'L0_S0_F2',
 'L0_S0_F4',
 'L0_S0_F6',
 'L0_S0_F8',
 'L0_S0_F10',
 'L0_S0_F12',
 'L0_S0_F14',
 'L0_S0_F16',
 'L0_S0_F18',
 'L0_S0_F20',
 'L0_S0_F22',
 'L0_S1_F24',
 'L0_S1_F28',
 'L0_S2_F32',
 'L0_S2_F36',
 'L0_S2_F40',
 'L0_S2_F44',
 'L0_S2_F48']

In [6]:
print(df_bosch['Response'].value_counts())
print(df_bosch['Response'].value_counts(normalize=True) * 100)

Response
0    117685
1       690
Name: count, dtype: int64
Response
0    99.417107
1     0.582893
Name: proportion, dtype: float64


In [7]:
lines = set()
for col in df_bosch.columns:
    if col.startswith('L'):
        line_id = col.split('_')[0]
        lines.add(line_id)

print(sorted(lines))

['L0', 'L1', 'L2', 'L3']


In [8]:
missing_pct = df_bosch.isnull().mean().mean() * 100
print(f"Pourcentage moyen de valeurs manquantes : {missing_pct:.2f}%")

Pourcentage moyen de valeurs manquantes : 80.91%


In [9]:
missing_per_col = df_bosch.isnull().mean()
cols_to_keep = missing_per_col[missing_per_col < 0.90].index.tolist()

df_bosch_reduced = df_bosch[cols_to_keep]
print(df_bosch_reduced.shape)

(118375, 334)


In [10]:
df_bosch_reduced = df_bosch_reduced.fillna(0)

In [11]:
X_bosch = df_bosch_reduced.drop(columns=['Id', 'Response'])
y_bosch = df_bosch_reduced['Response']

print(X_bosch.shape)

(118375, 332)


In [12]:
from sklearn.model_selection import train_test_split

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bosch, y_bosch,
    test_size=0.2,
    random_state=42,
    stratify=y_bosch
)

print(X_train_b.shape, X_test_b.shape)
print(y_train_b.value_counts())

(94700, 332) (23675, 332)
Response
0    94148
1      552
Name: count, dtype: int64


In [13]:
from xgboost import XGBClassifier

# Modèle rapide, juste pour classer les features par importance
selector_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=(y_train_b.value_counts()[0] / y_train_b.value_counts()[1])
)
selector_model.fit(X_train_b, y_train_b)

feature_importance = pd.Series(
    selector_model.feature_importances_,
    index=X_bosch.columns
).sort_values(ascending=False)

top_features = feature_importance.head(80).index.tolist()
print(feature_importance.head(20))

L3_S33_F3857    0.016893
L3_S30_F3799    0.014435
L3_S35_F3896    0.012210
L0_S9_F195      0.011124
L0_S12_F330     0.011018
L0_S12_F348     0.009329
L3_S29_F3430    0.008886
L3_S30_F3679    0.008243
L3_S30_F3609    0.008216
L3_S30_F3644    0.008185
L0_S7_F142      0.008169
L2_S26_F3121    0.008145
L3_S33_F3865    0.008109
L1_S24_F1844    0.007942
L1_S24_F1829    0.007782
L0_S14_F362     0.007674
L0_S11_F290     0.007666
L3_S30_F3634    0.007558
L1_S24_F1518    0.007361
L3_S30_F3604    0.007350
dtype: float32


In [14]:
X_train_b_reduced = X_train_b[top_features]
X_test_b_reduced = X_test_b[top_features]

print(X_train_b_reduced.shape, X_test_b_reduced.shape)

(94700, 80) (23675, 80)


In [15]:
from xgboost import XGBClassifier
from sklearn.metrics import matthews_corrcoef, classification_report

scale_pos = y_train_b.value_counts()[0] / y_train_b.value_counts()[1]

xgb_bosch = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric='logloss'
)
xgb_bosch.fit(X_train_b_reduced, y_train_b)

y_pred_bosch = xgb_bosch.predict(X_test_b_reduced)

print(classification_report(y_test_b, y_pred_bosch))
print("MCC:", matthews_corrcoef(y_test_b, y_pred_bosch))

              precision    recall  f1-score   support

           0       1.00      0.92      0.95     23537
           1       0.02      0.22      0.03       138

    accuracy                           0.91     23675
   macro avg       0.51      0.57      0.49     23675
weighted avg       0.99      0.91      0.95     23675

MCC: 0.03685585853051769


In [17]:
from lightgbm import LGBMClassifier

lgbm_bosch = LGBMClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    random_state=42,
    verbose=-1
)
lgbm_bosch.fit(X_train_b_reduced, y_train_b)

y_pred_lgbm = lgbm_bosch.predict(X_test_b_reduced)

print(classification_report(y_test_b, y_pred_lgbm))
print("MCC:", matthews_corrcoef(y_test_b, y_pred_lgbm))

c:\Users\PC\Desktop\apex\venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Le fichier spécifié est introuvable
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\PC\Desktop\apex\venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "c:\Users\PC\Desktop\apex\venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 554, in r

              precision    recall  f1-score   support

           0       1.00      0.91      0.95     23537
           1       0.02      0.26      0.03       138

    accuracy                           0.91     23675
   macro avg       0.51      0.59      0.49     23675
weighted avg       0.99      0.91      0.95     23675

MCC: 0.04527503659430522


In [18]:
from sklearn.metrics import matthews_corrcoef
import numpy as np

# Probabilités plutôt que prédiction binaire directe
y_proba = xgb_bosch.predict_proba(X_test_b_reduced)[:, 1]

thresholds = np.arange(0.01, 0.99, 0.01)
mcc_scores = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    mcc_scores.append(matthews_corrcoef(y_test_b, y_pred_t))

best_idx = np.argmax(mcc_scores)
best_threshold = thresholds[best_idx]
best_mcc = mcc_scores[best_idx]

print(f"Meilleur seuil : {best_threshold:.2f}")
print(f"MCC optimal : {best_mcc:.4f}")

# Rapport avec le seuil optimal
y_pred_optimal = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test_b, y_pred_optimal))

Meilleur seuil : 0.84
MCC optimal : 0.1315
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     23537
           1       0.31      0.06      0.10       138

    accuracy                           0.99     23675
   macro avg       0.65      0.53      0.55     23675
weighted avg       0.99      0.99      0.99     23675



In [19]:
import joblib

joblib.dump(xgb_bosch, '../models/module_b_gonogo_classifier.pkl')
joblib.dump(top_features, '../models/module_b_selected_features.pkl')
joblib.dump(best_threshold, '../models/module_b_optimal_threshold.pkl')

print("Modèle, features sélectionnées et seuil optimal sauvegardés.")

Modèle, features sélectionnées et seuil optimal sauvegardés.
